# Feature block selection through temporal validation

This example compares cumulative feature blocks using the **same forecast origins and horizons** with `MLForecast.cross_validation`. The forecast horizon is 7 days, and the library generates lag features recursively. Calendar and promotion information are assumed to be known at forecast time.

In [1]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path

from mlforecast import MLForecast
from sklearn.ensemble import HistGradientBoostingRegressor

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
sys.path.insert(0, str(project_root))
from tinyshift.forecasting import fva, pbias, rmae, score, wape

## Sample data

Promotions have a real effect on demand, whereas `external_noise` does not. With real data, each exogenous column should have an explicit availability rule, typically enforced through an as-of join.

In [2]:
rng = np.random.default_rng(42)
n = 420
df = pd.DataFrame({"unique_id": "item_1", "ds": pd.date_range("2025-01-01", periods=n)})
dow = df["ds"].dt.dayofweek
df["promo"] = rng.binomial(1, 0.12, n)
df["external_noise"] = rng.normal(size=n)
weekly = 8 * np.sin(2 * np.pi * dow / 7)
df["y"] = 50 + weekly + 15 * df["promo"] + rng.normal(0, 4, n)
df.head()

,unique_id,ds,promo,external_noise,y
0,item_1,2025-01-01,0,0.521167,50.352043
1,item_1,2025-01-02,0,-0.265839,53.587510
2,item_1,2025-01-03,0,-0.117542,46.652599
3,item_1,2025-01-04,0,0.829519,41.730133
4,item_1,2025-01-05,0,-1.993060,48.602107


## Temporal validation with MLForecast

The feature blocks are cumulative. `cross_validation` creates the windows, refits the estimator in each one (`refit=True`), and returns the `cutoff`, actual values, and forecasts. Passing `static_features=[]` indicates that the additional columns vary over time; consequently, their future values must be available at forecast time.

In [3]:
feature_blocks = {
    "baseline": [],
    "with_promo": ["promo"],
    "with_external": ["promo", "external_noise"],
}
horizon, n_windows, step_size = 7, 7, 28
keys = ["unique_id", "ds", "cutoff", "y"]
predictions = None

for name, exogenous in feature_blocks.items():
    fcst = MLForecast(
        models={name: HistGradientBoostingRegressor(max_iter=150, max_depth=3, random_state=42)},
        freq="D",
        lags=[14, 28],
        date_features=["dayofweek"],
    )
    cv = fcst.cross_validation(
        df=df[["unique_id", "ds", "y", *exogenous]],
        n_windows=n_windows,
        h=horizon,
        step_size=step_size,
        static_features=[],
        refit=True,
    )
    predictions = cv if predictions is None else predictions.merge(cv[keys + [name]], on=keys, validate="one_to_one")

assert len(predictions) == n_windows * horizon
predictions.head(10)

,unique_id,ds,cutoff,y,baseline,with_promo,with_external
0,item_1,2025-09-03,2025-09-02,57.489999,58.465214,59.031747,57.655752
1,item_1,2025-09-04,2025-09-02,52.683527,57.764693,54.881286,54.931703
2,item_1,2025-09-05,2025-09-02,42.070462,51.507023,46.442599,46.667701
3,item_1,2025-09-06,2025-09-02,41.283406,42.929230,42.463610,43.285763
4,item_1,2025-09-07,2025-09-02,37.374171,43.509284,42.748367,42.426932
5,item_1,2025-09-08,2025-09-02,46.348488,51.042423,49.378417,48.575942
6,item_1,2025-09-09,2025-09-02,57.161796,62.545039,59.798532,60.043112
7,item_1,2025-10-01,2025-09-30,59.777650,59.459119,58.618385,58.941251
8,item_1,2025-10-02,2025-09-30,57.365724,55.292073,52.905970,53.313723
9,item_1,2025-10-03,2025-09-30,51.496770,46.415234,44.271059,45.262451


## Comparison

RMAE compares MAE, whereas FVA compares `score = WAPE + |PBias|`. Both use `baseline` as their common reference. A production decision should also inspect results by forecast origin, horizon, and relevant groups instead of relying only on aggregate performance.

In [4]:
candidates = ["with_promo", "with_external"]
all_models = ["baseline", *candidates]
metric_frames = [
    wape(predictions, all_models),
    pbias(predictions, all_models),
    score(predictions, all_models),
    rmae(predictions, candidates, baseline_col="baseline"),
    fva(predictions, candidates, baseline_col="baseline"),
]
summary = (
    pd.concat(metric_frames, ignore_index=True)
    .drop(columns="unique_id")
    .set_index("metric")
    .T
)
summary

metric,wape,pbias,score,rmae,fva
baseline,0.102110,-0.002895,0.105004,NaN,NaN
with_promo,0.074853,-0.002219,0.077072,0.733062,0.266012
with_external,0.071939,0.000617,0.072556,0.704531,0.309016


A configuration should be retained when its improvement is consistent out of sample and operationally relevant. In this example, promotions are expected to add value; the noise block should not be retained merely because it happens to improve aggregate performance by chance. To measure the incremental contribution of each stage, use the immediately preceding configuration as the reference. The FVA shown above is cumulative relative to the common baseline.